In [ ]:
import sys
sys.path.insert(0, "/workspace/app")
import pandas as pd
import random
import shap
import numpy as np
import os
import ast
import pickle
from sklearn.preprocessing import LabelEncoder
from coding.Data_Preparation import Get_embSessions as dp
from coding.Data_Preprocessing import DataPreprocessing_melt as mf
from scipy.stats import spearmanr
from coding.Data_Processing import TargetVariables as tv
encoder = LabelEncoder()
MAX_HDRS = 69 
MAX_CDI = 90  
lower_pct = 40
upper_pct = 60

> Note: This script is used to create the "df_all_Horizon2" dataset used in the experiments.

Load initial data

In [ ]:
def depression_standardized_T0(df_model, MAX_HDRS, MAX_CDI):
    df_model.insert(len(df_model.columns), 'Y_Depression_T0', None)
    df_model['Y_Depression_T0'] = df_model['Y_Depression_T0'].astype(float)
    for line_index in df_model.index:
        if pd.notna(df_model.loc[line_index, 'Questionnary1_HDRS_T0_TOT21_Score']):
            df_model.loc[line_index, 'Y_Depression_T0'] = df_model.loc[line_index, 'Questionnary1_HDRS_T0_TOT21_Score'] / MAX_HDRS
        elif pd.notna(df_model.loc[line_index, 'Questionnary2_CDI_T0_Score']):
            df_model.loc[line_index, 'Y_Depression_T0'] = (df_model.loc[line_index, 'Questionnary2_CDI_T0_Score'] - 40) / (MAX_CDI - 40)
    return df_model

Patients that did NOT drop out

In [ ]:
df_mfcc = pd.read_pickle("/workspace/app/planilhas/df_mfccs_mean_NotNorm.pkl")
df_emb = pd.read_pickle("/workspace/app/planilhas/df_emb3D.pkl")

Patients that dropped out

In [ ]:
df_mfcc = pd.read_pickle("/workspace/app/planilhas/df_mfccs_dropout.pkl")
df_emb = pd.read_pickle("/workspace/app/planilhas/df_emb3D_dropout.pkl")

_____

Merge data

In [ ]:
merged = df_mfcc.merge(df_emb, on="Patient_ID", how="outer", suffixes=("_df_mfcc", "_df_emb"))
common = (set(df_mfcc.columns) & set(df_emb.columns)) - {"Patient_ID"}
for col in common:
    col_left = f"{col}_df_mfcc"
    col_right = f"{col}_df_emb"
    merged[col] = merged[col_left].combine_first(merged[col_right])
    merged = merged.drop(columns=[col_left, col_right])
df = merged
df['Patient_ID'] = encoder.fit_transform(df['Patient_ID'])
df = df.rename(columns={"Embedding_Mean": "Embedding_npy"})
print(f"All Patients: {df['Patient_ID'].nunique()}")

In [ ]:
df = depression_standardized_T0(df, MAX_HDRS, MAX_CDI)
df = tv.zscore_TO_T1_TOT_Alexithymia(df)

In [ ]:
# Columns that won't be used
date_subscore_data = [col for col in df.columns if col.endswith('Date') or 
    col.endswith('HQ_25_T0_TOT_Score') or col.endswith('HQ_25_T1_TOT_Score') or 
    col.endswith('F1_Score') or col.endswith('F2_Score') or col.endswith('F3_Score')] 
df.drop(columns=date_subscore_data, inplace=True)
mfcc_std_drop = [
    f"MFCC_Session{s}_Segment{i}_StdCoefficient{j}"
    for s in range(1, 9)         # sessions
    for i in range(0, 700)       # segments
    for j in range(1, 14) ]      # coeficients
df = df.drop(columns=mfcc_std_drop, errors="ignore")
# For MFCC use _Coefficient instead of _MeanCoefficient
df = df.rename(columns=lambda c: c.replace("MeanCoefficient", "Coefficient"))

In [ ]:
# Melt df - patient sessions as lines
meta = ['Patient_ID', 'Age', "Patient_Gender", 'Education', 
        'Y_Zscore_TO_TOT_Alexithymia', 'Questionnary5_HQ_25_T0_soc_Score', 'Questionnary5_HQ_25_T0_iso_Score', 'Questionnary5_HQ_25_T0_sup_Score',               
        'Mean_Gap_Days', 'Std_Gap_Days', 'Y_Depression_T0']
df = mf.get_f0_mfccs_embeddings_per_session_mean_std(df, meta)
print(f"Patients after melting df: {df['Patient_ID'].nunique()}") 

In [ ]:
# Organizing DF
new_order = [c for c in df.columns if c not in ["Y_Depression_T0"]]
new_order += ["Y_Depression_T0"]
df = df[new_order]

In [ ]:
mfcc_cols = [c for c in df.columns if c.startswith("MFCC") and c.endswith("_mean")]
mfcc_cols = sorted(mfcc_cols, key=lambda c: int(c.replace("MFCC", "").replace("_mean", "")))

Normalize

In [ ]:
df = dp.normalize_intra_patient(df, mfcc_cols)

In [ ]:
mfcc_std_drop = [
    f"MFCC{j}_std"
    for j in range(1, 14)]  
df = df.drop(columns=mfcc_std_drop, errors="ignore")

In [ ]:
# Patients that didn't drop out - Using only 35 patients [the subset used in the CBMS related publication]
yaa_remover = [0, 12, 13, 14, 15, 16, 17, 18, 19, 20, 27, 
               29, 32, 35, 36, 37, 51, 52]
df = df[~df["Patient_ID"].isin(yaa_remover)]
print(f"Patients: {df['Patient_ID'].nunique()}")
#df.to_excel('/workspace/app/coding/Paper_CBMS/df_notdropout.xlsx', index=False)

In [ ]:
# For patients that dropped out [the subset used in the CBMS related publication]
df['Patient_ID'] = encoder.fit_transform(df['Patient_ID']) + 100
#df.to_excel('/workspace/app/coding/Paper_CBMS/df_dropout.xlsx', index=False)

Structure data - Gropu the 5-seconds segments' sets (npy files) into their respective session. Result: npy embeddings files per sessions.  

In [ ]:
out_root="/workspace/app/embeddings_session_all"
list_col="Embedding_npy_list"
df = dp.materialize_session_embeddings(df, out_root, list_col, False)

_____

Merge Datasets

In [4]:
df_notdrop = pd.read_excel("/workspace/app/planilhas/CBMS/df_notdropout.xlsx")
df_drop = pd.read_excel("/workspace/app/planilhas/CBMS/df_dropout.xlsx")

In [5]:
df_final = pd.concat([df_notdrop, df_drop], ignore_index=True)

___

Create Hazard DF

In [ ]:
def create_hazard_df(df):
    last_session = df.groupby("Patient_ID")["Session"].max().reset_index()
    last_session.columns = ["Patient_ID", "Last_session"]
    df = df.merge(last_session, on="Patient_ID", how="left")

    df["Dropout_session"] = np.where(df["Last_session"] < 8, df["Last_session"], np.nan)

    # y_next = 1 if dropout happens in the next session
    df["y_next"] = ((df["Dropout_session"].notna()) & (df["Session"] + 1 == df["Dropout_session"])).astype(int)

    # Remove the last session (it's impossible to predict the "next" session)
    hazard_df = df[df["Session"] < df["Last_session"]].copy()

    # Remove columns that can cause leakage
    hazard_df = hazard_df.drop(columns=["Last_session", "Dropout_session"])
    return hazard_df

In [ ]:
df = create_hazard_df(df)
df.to_excel('/workspace/app/coding/Paper_CBMS/df_all_Hazard.xlsx', index=False)

___

Create Horizon-2 (y_h2)

In [ ]:
def creat_horizon2(df):
    df["Session"] = df["Session"].astype(int)

    # dropout_session: last observed session if < 8 else NaN (completed)
    last_sess = df.groupby("Patient_ID")["Session"].max().rename("last_session")
    df = df.merge(last_sess, on="Patient_ID", how="left")
    df["dropout_session"] = np.where(df["last_session"] < 8, df["last_session"], np.nan)

    # horizon-2: dropout at t+1 OR t+2
    df["y_h2"] = (
        df["dropout_session"].notna() &
        (
            (df["Session"] + 1 == df["dropout_session"]) |
            (df["Session"] + 2 == df["dropout_session"])
        )
    ).astype(int)
    # remove helper cols (avoid leakage)
    df = df.drop(columns=["last_session", "dropout_session"])
    return df

In [ ]:
df = creat_horizon2(df_final)
df.to_excel('/workspace/app/planilhas/CBMS/df_all_Horizon2.xlsx', index=False)

____

Create therapy attendance derived features - Engagement markers

In [ ]:
def engagement_long_from_wide(df):
    rows = []
    for _, r in df.iterrows():
        pid = r["Patient_ID"]
        dates = [r.get(f"Recording{i}_Date", pd.NaT) for i in range(1, 9)]
        s1_date = dates[0]
        gaps = []
        for t in range(1, 9):  # session t
            dt = dates[t-1]
            if pd.isna(dt):
                continue
            # elapsed_time_so_far(t)
            elapsed = (dt - s1_date).days
            if t == 1:
                mean_gap = 0.0
                std_gap = 0.0
            else:
                gap_col = f"Days_Between_Sessions_{t-1}and{t}"
                gap_val = r.get(gap_col, np.nan)
                if pd.notna(gap_val):
                    gaps.append(float(gap_val))
                # cumulative untill t
                mean_gap = float(np.mean(gaps)) if len(gaps) else 0.0
                std_gap  = float(np.std(gaps))  if len(gaps) else 0.0
            rows.append({
                "Patient_ID": pid,
                "Session": int(t),
                "elapsed_time_so_far": float(elapsed),
                "mean_gap_so_far": float(mean_gap),
                "std_gap_so_far": float(std_gap),
            })
    return pd.DataFrame(rows)

In [ ]:
from coding.Data_Preparation import Tabular_Data as td
td.set_days_between_sessions(df)
td.set_complete_sessions(df)
df_eng = engagement_long_from_wide(df)